## TO do
POSITIONAL ENCODING

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
import torch.nn as nn
import torch
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),                  
    transforms.Normalize((0.5,), (0.5,)),     
])

train_set = datasets.MNIST(
    root="./data",        
    train=True,
    download=True,
    transform=transform,
)

test_set = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=128, shuffle=False, num_workers=0)


100%|██████████| 9.91M/9.91M [00:00<00:00, 137MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 37.7MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 78.3MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.14MB/s]


In [3]:
train_set[0][0].shape

torch.Size([1, 28, 28])

In [4]:
class PatchEmbedding(nn.Module):
    """
    A layer that splits an image into patches and projects each patch to an embedding vector.
    Used as the input layer of a Vision Transformer (ViT).

    Inputs:
    - img_size: Integer representing the height/width of input image (assumes square image).
    - patch_size: Integer representing height/width of each patch (square patch).
    - in_channels: Number of input image channels
    - embed_dim: Dimension of the linear embedding space.
    """
    def __init__(self, img_size, patch_size, in_channels=1, embed_dim=64):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        assert img_size%patch_size==0, "img dim must be divisible by patch_size"
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_dim = in_channels * patch_size * patch_size
        self.proj = nn.Linear(self.patch_dim, embed_dim)
        self.embed_dim = embed_dim

    def forward(self, x):
        """
        input: x.shape = (B, C, H, W); C == in_channels
        
        patch_dim = C* patch_size * patch_size
        num_patches = (H//p) * (W//p)

        output: patchified img: (B, num_patches, embed_dim)
        """
        B, C, _, _ = x.shape
        p = self.patch_size
        x = x.view(B, C,  self.img_size//p, self.img_size//p, p, p)
        # print(x.is_contiguous())
        x = x.permute(0,3,1,2,4,5)
        # print(x.is_contiguous())

        x = x.reshape(B, self.num_patches, self.patch_dim)
        # print(x.is_contiguous())
        return self.proj(x)

In [5]:
## test
x = torch.randn(5, 1, 28, 28)
patching_test = PatchEmbedding(28, 7, 1, 64)
out = patching_test(x)
out.shape

torch.Size([5, 16, 64])

In [ ]:
import math
def embedsinus(dim, x):
    """
    x -> (B,)
    """
    N=10000 #Attention is all you need  N**(k/(dim//2)) 
    h_dim = dim//2
    emb = math.log(N)/(h_dim)
    emb = torch.exp(torch.arange(h_dim) * -emb) # (h_dim,)  
    emb = x[:, None] * emb[None,:] #careful with broadcast
    return torch.cat((emb.sin(), emb.cos()), dim=-1)

def pe2d(num_patch, emb_dim):
    grid_size = int(num_patch**0.5)
    rows = torch.arange(grid_size)
    cols = torch.arange(grid_size)
    assert grid_size**2==num_patch, "num_patch has to be a square"
    assert emb_dim%4==0, "emb_dim has to be divisible by 4"
    rows = embedsinus(emb_dim // 2, rows)
    cols = embedsinus(emb_dim // 2, cols) 
    # what we do here is a bit more trickier than 1D 
    # we do a grid of positional encoding (col(i), row(j)) for the matrix/grid
    #we then flatten to get (1, num_patch, emb_dim)
    pe_r = rows[:, None, :].expand(grid_size, grid_size, emb_dim // 2)
    pe_c = cols[None, :, :].expand(grid_size, grid_size, emb_dim // 2)
    pe = torch.cat([pe_r, pe_c], dim=-1)
    print(pe.shape)
    return pe.view(1, num_patch, emb_dim)
embedsinus(16, torch.randn(5)).shape
# pe2d(9,16)

torch.Size([5, 16])